# QCD estimation

In [1]:
%cd ..
%load_ext autoreload
%autoreload 2
from pathlib import Path
import yaml
import numpy as np
import matplotlib.pyplot as plt
import mplhep as hep
from coffea.processor import accumulate
from notebooks.utils import *
from coffea.util import load
import numpy as np
import mplhep as hep
from matplotlib.offsetbox import AnchoredText
import matplotlib.pyplot as plt
from hist.intervals import poisson_interval
from pathlib import Path
import yaml
from analysis.filesets.utils import get_process_maps
from analysis.workflows.config import WorkflowConfigBuilder
import json
import hist
import yaml
import numpy as np
import collections
from scipy.optimize import curve_fit
from scipy.stats import f as f_dist
import matplotlib.text as mtext
import matplotlib.ticker as ticker
import matplotlib.gridspec as gridspec
hep.style.use(hep.style.CMS)
import sys

for module in list(sys.modules.keys()):
    if module.startswith("notebooks.utils"):
        del sys.modules[module]

/afs/cern.ch/user/d/driosper/bsm3g_coffea


In [2]:
from notebooks.utils import generate_qcd_tf_tables_and_pngs
years = ["2022preEE", 
         #"2022postEE", "2023preBPix", "2023postBPix"
         ]
generate_qcd_tf_tables_and_pngs(years=years)

Loading histograms for qcd_tf_mu | Year=2022preEE...
  Processing table: Region=2T | Workflow=qcd_tf_mu | Year=2022preEE...
  ✓ PNG saved successfully: notebooks/plots/qcd_tf_estimation/table_qcd_tf_mu_2T_2022preEE.png
  Processing table: Region=1T1LnT | Workflow=qcd_tf_mu | Year=2022preEE...
  ✓ PNG saved successfully: notebooks/plots/qcd_tf_estimation/table_qcd_tf_mu_1T1LnT_2022preEE.png
Loading histograms for qcd_tf_ele | Year=2022preEE...


FileNotFoundError: [Errno 2] No such file or directory: '/eos/user/d/driosper/bsm3g_coffea_outputs/qcd_tf_ele/2022preEE/2022preEE_processed_histograms.coffea'

In [3]:
from notebooks.utils import generate_qcd_cr_tables_and_pngs
years = ["2022preEE", "2022postEE", "2023preBPix", "2023postBPix"]
generate_qcd_cr_tables_and_pngs(years=years)

Loading histograms for qcd_ele | Year=2022preEE...
  Processing table: Region=base | Workflow=qcd_ele | Year=2022preEE...
  ✓ PNG saved successfully: notebooks/plots/qcd_control_region/table_qcd_ele_base_2022preEE.png
Loading histograms for qcd_ele | Year=2022postEE...
  Processing table: Region=base | Workflow=qcd_ele | Year=2022postEE...
  ✓ PNG saved successfully: notebooks/plots/qcd_control_region/table_qcd_ele_base_2022postEE.png
Loading histograms for qcd_ele | Year=2023preBPix...
  Processing table: Region=base | Workflow=qcd_ele | Year=2023preBPix...
  ✓ PNG saved successfully: notebooks/plots/qcd_control_region/table_qcd_ele_base_2023preBPix.png
Loading histograms for qcd_ele | Year=2023postBPix...
  Processing table: Region=base | Workflow=qcd_ele | Year=2023postBPix...
  ✓ PNG saved successfully: notebooks/plots/qcd_control_region/table_qcd_ele_base_2023postBPix.png
Loading histograms for qcd_mu | Year=2022preEE...
  Processing table: Region=base | Workflow=qcd_mu | Year=202

### QCD TRANSFER FACTOR

In [3]:
apply_cms_text_patch()
hep.style.use("CMS")

plt.rcParams.update({
    "font.size": 12,
    "axes.labelsize": 14,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
})

energy = {"2": "13", "3": "13.6"}
postprocess_dir = Path.cwd() / "analysis" / "postprocess"
with open(postprocess_dir / "luminosity.yaml", "r") as f:
    luminosities = yaml.safe_load(f)

plots_dir = Path("notebooks/plots/qcd_estimation")
plots_dir.mkdir(parents=True, exist_ok=True)

years = ["2022preEE", #"2022postEE", "2023preBPix", "2023postBPix"
         ]
workflows = ["qcd_tf_mu", #"qcd_tf_mu"
             ]

labels_map = {
    "qcd_tf_ele": r"$m_T(e_1, p_T^{\mathrm{miss}})$ [GeV]",
    "qcd_tf_mu": r"$m_T(\mu_1, p_T^{\mathrm{miss}})$ [GeV]",
}

tf_store = {}

for workflow in workflows:
    fig, axes = plt.subplots(2, 2, figsize=(14, 12), sharex=False, sharey=False)
    grid_indices = [(0, 0), (0, 1), (1, 0), (1, 1)]
    tf_store[workflow] = {}

    for year, (row, col) in zip(years, grid_indices):
        run_key = "2" if year.startswith("201") else "3"
        ax_grid = axes[row, col]
        
        h_Tight, h_nTight = compute_tf_for_year(year, workflow)
        edges = h_Tight.axes[0].edges
        centers = 0.5 * (edges[:-1] + edges[1:])
        widths = np.diff(edges)
        xerr = widths / 2.0
        
        num = h_Tight.values()
        den = h_nTight.values() + 1e-32
        
        var_num = h_Tight.variances() if h_Tight.variances() is not None else np.abs(num)
        var_den = h_nTight.variances() if h_nTight.variances() is not None else np.abs(den)
        
        values = num / den
        ratio_variance = (var_num / (den**2)) + ((num**2 * var_den) / (den**4))
        yerr_for_fit = np.sqrt(np.maximum(ratio_variance, 0))
        ratio_uncert = np.array([yerr_for_fit, yerr_for_fit])

        valid_mask = (
            (values >= 0)
            & (yerr_for_fit > 0)
            & ~np.isnan(values)
            & ~np.isnan(yerr_for_fit)
            & ~np.isinf(values)
            & ~np.isinf(yerr_for_fit)
        )
        
        x_fit = centers[valid_mask]
        y_fit = values[valid_mask]
        yerr_fit = yerr_for_fit[valid_mask]
        
        best_degree, optimal_coefs, _ = find_optimal_tf_poly(x_fit, y_fit, yerr_fit)
        
        tf_smooth_values = np.maximum(poly_func(centers, *optimal_coefs), 1e-5)
        tf_store[workflow][year] = tf_smooth_values

        x_fine = np.linspace(edges[0], edges[-1], 200)
        y_fine = np.maximum(poly_func(x_fine, *optimal_coefs), 1e-5)

        lumi_val = float(round(luminosities[year] * 1e-3, 1))
        com_val = float(energy[run_key].split()[0])
        xlabel_str = labels_map[workflow]

        tf_pack = (centers, values, xerr, ratio_uncert, edges)
        fit_pack = (x_fine, y_fine, best_degree)
        meta_pack = (xlabel_str, lumi_val, com_val, year)

        plot_tf_panel(ax_grid, tf_pack, fit_pack, meta_pack)

        fig_ind, ax_ind = plt.subplots(figsize=(8, 6))
        plot_tf_panel(ax_ind, tf_pack, fit_pack, meta_pack)
        fig_ind.tight_layout()
        fig_ind.savefig(
            plots_dir / f"qcd_tf_agreement_{workflow}_{year}.png",
            dpi=300,
            bbox_inches="tight",
        )
        plt.close(fig_ind)

    # Guardado del panel general
    fig.tight_layout()
    fig.savefig(plots_dir / f"qcd_tf_agreement_{workflow}.png", dpi=300, bbox_inches="tight")
    plt.close(fig)

### INJECTION OF QCD EVENTS INTO SIGNAL REGION

In [5]:
from pathlib import Path
import yaml
import mplhep as hep
from notebooks.utils import apply_cms_text_patch, plot_qcd_injected_signals, generate_qcd_tables_and_pngs, run_qcd_estimation

apply_cms_text_patch()
hep.style.use("CMS")

postprocess_dir = Path.cwd() / "analysis" / "postprocess"
with open(postprocess_dir / "luminosity.yaml", "r") as f:
    luminosities = yaml.safe_load(f)

years = ["2022preEE", "2022postEE", "2023preBPix", "2023postBPix"]
plots_dir = Path("notebooks/plots/qcd_estimation")

qcd_store = run_qcd_estimation(tf_store, years, luminosities, plots_dir)
plot_qcd_injected_signals(qcd_store, years, luminosities, plots_dir)
generate_qcd_tables_and_pngs(qcd_store, years, plots_dir)

Processing table: SR=1b1e | Year=2022preEE...
  ✓ PNG saved successfully: notebooks/plots/qcd_estimation/table_1b1e_2022preEE_qcd.png
Processing table: SR=1b1e | Year=2022postEE...
  ✓ PNG saved successfully: notebooks/plots/qcd_estimation/table_1b1e_2022postEE_qcd.png
Processing table: SR=1b1e | Year=2023preBPix...
  ✓ PNG saved successfully: notebooks/plots/qcd_estimation/table_1b1e_2023preBPix_qcd.png
Processing table: SR=1b1e | Year=2023postBPix...
  ✓ PNG saved successfully: notebooks/plots/qcd_estimation/table_1b1e_2023postBPix_qcd.png
Processing table: SR=1b1mu | Year=2022preEE...
  ✓ PNG saved successfully: notebooks/plots/qcd_estimation/table_1b1mu_2022preEE_qcd.png
Processing table: SR=1b1mu | Year=2022postEE...
  ✓ PNG saved successfully: notebooks/plots/qcd_estimation/table_1b1mu_2022postEE_qcd.png
Processing table: SR=1b1mu | Year=2023preBPix...
  ✓ PNG saved successfully: notebooks/plots/qcd_estimation/table_1b1mu_2023preBPix_qcd.png
Processing table: SR=1b1mu | Year=2023